<a href="https://colab.research.google.com/github/Cautioncrazy/kohya_ss_LoRA-Trainer/blob/master/kohya_ss_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Train with Kohya's Stable Diffusion Trainers
%cd /content

from google.colab import drive
drive.mount('/content/drive')

!pip install dadaptation==3.1 diffusers[torch]==0.17.1 easygui==0.98.3 einops==0.6.0 fairscale==0.4.13 ftfy==6.1.1 gradio==3.36.1 huggingface-hub==0.14.1
!pip install lion-pytorch==0.0.6 lycoris_lora==1.8.0.dev6 open-clip-torch==2.20.0 prodigyopt==1.0 pytorch-lightning==1.9.0 safetensors==0.3.1 timm==0.6.12
!pip install tk==0.1.0 transformers==4.30.2 voluptuous==0.13.1 wandb==0.15.0 xformers==0.0.20 omegaconf

%cd /content
!git clone -b 0.41.0 https://github.com/TimDettmers/bitsandbytes
%cd /content/bitsandbytes
!CUDA_VERSION=118 make cuda11x
!python setup.py install

%cd /content
!git clone -b v1.0 https://github.com/camenduru/kohya_ss
%cd /content/kohya_ss

!python kohya_gui.py --share --headless

Run Kohya Gui

In [ ]:
# 1. Clean up and install with a stable configuration to prevent the torch-reinstall loop
!pip install --upgrade pip
!pip install torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install xformers separately without dependencies to prevent it from downgrading torch
!pip install xformers==0.0.28.post3 --no-deps

# Install remaining requirements
!pip install accelerate==0.32.1 huggingface-hub==0.25.0 diffusers==0.25.0
!pip install transformers==4.44.0 safetensors==0.4.2 bitsandbytes==0.41.3.post2
!pip install gradio==3.36.1 easygui==0.98.3 einops==0.6.0 voluptuous==0.13.1
!pip install open-clip-torch==2.20.0 tensorboard==2.15.0

# 2. Launch with environment variables and Low-RAM fix
%cd /content/kohya_ss
import os
import torch

# Forces torch to be more aggressive with memory reuse
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'

# Pre-clear memory before starting
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Launching with --lowram to ensure the trainer doesn't spike System RAM during load
!python kohya_gui.py --share --headless

In [38]:
#@title Direct SDXL Training Command (VRAM-Load Strategy)
import os
import torch
import gc

# 1. System Deep Clean
gc.collect()
torch.cuda.empty_cache()

# 2. Setup Paths
%cd /content/kohya_ss

# 3. Apply the Direct-to-GPU Patch
# This forces the loader to use 'cuda' immediately, bypassing the System RAM (CPU) stage
!sed -i "s/load_file(checkpoint_path)/load_file(checkpoint_path, device='cuda')/g" library/sdxl_model_util.py
!sed -i "s/load_file(model_path)/load_file(model_path, device='cuda')/g" library/model_util.py

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'

MODEL_PATH = "/content/ponyDiffusionV6XL.safetensors"
DATA_DIR = "/content/drive/Othercomputers/My Laptop/pokemon_lora_project"
OUTPUT_DIR = "/content/drive/Othercomputers/My Laptop/pokemon_lora_project/Outputs/pkmnessentialitem"
LOG_DIR = "/content/drive/Othercomputers/My Laptop/pokemon_lora_project/log"
OUTPUT_NAME = "pkmn_items_v1"

# 4. Launch with Low-RAM strategies and 512 resolution
!accelerate launch --num_cpu_threads_per_process=1 "./sdxl_train_network.py" \
    --enable_bucket \
    --pretrained_model_name_or_path="{MODEL_PATH}" \
    --train_data_dir="{DATA_DIR}" \
    --resolution="512,512" \
    --output_dir="{OUTPUT_DIR}" \
    --logging_dir="{LOG_DIR}" \
    --network_alpha=16 \
    --save_model_as=safetensors \
    --network_module=networks.lora \
    --network_dim=32 \
    --output_name="{OUTPUT_NAME}" \
    --lr_scheduler_num_cycles=1 \
    --lr_scheduler_power=1 \
    --no_half_vae \
    --learning_rate=0.0001 \
    --unet_lr=0.0001 \
    --network_train_unet_only \
    --lr_scheduler=cosine \
    --lr_warmup_steps=200 \
    --train_batch_size=1 \
    --max_train_steps=2000 \
    --save_every_n_epochs=2 \
    --mixed_precision=fp16 \
    --save_precision=fp16 \
    --seed=0 \
    --caption_extension=.txt \
    --optimizer_type=AdamW8bit \
    --bucket_reso_steps=64 \
    --gradient_checkpointing \
    --xformers \
    --bucket_no_upscale \
    --lowram \
    --mem_eff_attn

/content/kohya_ss
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.

In [37]:
import psutil
import torch

def print_memory_stats():
    # System RAM
    ram = psutil.virtual_memory()
    print(f"System RAM Usage: {ram.used / 1024**3:.2f} GB / {ram.total / 1024**3:.2f} GB ({ram.percent}%)")

    # GPU RAM
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            total_gpu = torch.cuda.get_device_properties(i).total_memory
            reserved_gpu = torch.cuda.memory_reserved(i)
            allocated_gpu = torch.cuda.memory_allocated(i)
            print(f"GPU {i} Usage: {allocated_gpu / 1024**3:.2f} GB / {total_gpu / 1024**3:.2f} GB")

print_memory_stats()

System RAM Usage: 1.21 GB / 12.67 GB (57.1%)
GPU 0 Usage: 0.00 GB / 14.56 GB


In [34]:
import os
import torch
from safetensors.torch import load_file
import psutil

def verify_and_monitor():
    model_path = "/content/ponyDiffusionV6XL.safetensors"
    if not os.path.exists(model_path):
        print(f"ERROR: Model not found at {model_path}")
        return

    print(f"Model size: {os.path.getsize(model_path) / 1024**3:.2f} GB")

    # Check RAM before loading
    ram = psutil.virtual_memory()
    print(f"Initial RAM: {ram.available / 1024**3:.2f} GB available")

    try:
        print("Testing model load to CPU (this spikes RAM)...")
        # Using mmap to see if we can avoid a full RAM load
        sd = load_file(model_path, device='cpu')
        print("Successfully read safetensors header.")
        del sd
        gc.collect()
    except Exception as e:
        print(f"Load test failed: {e}")

verify_and_monitor()

Model size: 6.46 GB
Initial RAM: 5.51 GB available
Testing model load to CPU (this spikes RAM)...
Successfully read safetensors header.


In [35]:
import torch
from safetensors.torch import load_file
import psutil

def check_vram_load(path='/content/ponyDiffusionV6XL.safetensors'):
    print(f'Checking if {path} can fit in VRAM...')
    initial_vram = torch.cuda.memory_reserved() / 1024**3
    initial_ram = psutil.virtual_memory().used / 1024**3

    try:
        # Load directly to GPU device
        state_dict = load_file(path, device='cuda')

        final_vram = torch.cuda.memory_reserved() / 1024**3
        final_ram = psutil.virtual_memory().used / 1024**3

        print(f'Success!')
        print(f'System RAM used: {final_ram - initial_ram:.2f} GB')
        print(f'GPU VRAM used: {final_vram - initial_vram:.2f} GB')

        del state_dict
        torch.cuda.empty_cache()
        print('VRAM cleared. If RAM usage stayed low, the VRAM-load strategy works.')
    except Exception as e:
        print(f'Load failed: {e}')

check_vram_load()

Checking if /content/ponyDiffusionV6XL.safetensors can fit in VRAM...
Success!
System RAM used: -0.03 GB
GPU VRAM used: 6.99 GB
VRAM cleared. If RAM usage stayed low, the VRAM-load strategy works.


In [21]:
import torch
from safetensors.torch import load_file
import psutil

def check_vram_load(path='/content/ponyDiffusionV6XL.safetensors'):
    print(f'Checking if {path} can fit in VRAM...')
    initial_vram = torch.cuda.memory_reserved() / 1024**3
    initial_ram = psutil.virtual_memory().used / 1024**3

    try:
        # Load directly to GPU device
        state_dict = load_file(path, device='cuda')

        final_vram = torch.cuda.memory_reserved() / 1024**3
        final_ram = psutil.virtual_memory().used / 1024**3

        print(f'Success!')
        print(f'System RAM used: {final_ram - initial_ram:.2f} GB')
        print(f'GPU VRAM used: {final_vram - initial_vram:.2f} GB')

        del state_dict
        torch.cuda.empty_cache()
        print('VRAM cleared. If RAM usage stayed low, the VRAM-load strategy works.')
    except Exception as e:
        print(f'Load failed: {e}')

check_vram_load()

Checking if /content/ponyDiffusionV6XL.safetensors can fit in VRAM...
Success!
System RAM used: 0.00 GB
GPU VRAM used: 6.99 GB
VRAM cleared. If RAM usage stayed low, the VRAM-load strategy works.


In [ ]:
#@title Convert Safetensors to Diffusers (Updated)
# Download a more recent version of the conversion script
!wget -q -O convert_diffusers.py https://raw.githubusercontent.com/huggingface/diffusers/main/scripts/convert_original_stable_diffusion_to_diffusers.py

# Convert Pony V6 (SDXL) to Diffusers format
# We remove the unrecognized --use_safetensors flag
!python3 convert_diffusers.py \
    --checkpoint_path /content/ponyDiffusionV6XL.safetensors \
    --dump_path /content/pony_diffusers \
    --from_safetensors \
    --to_safetensors \
    --device cuda

print("\nConversion complete! In Kohya GUI, use '/content/pony_diffusers' as your source model path.")

In [16]:
import psutil
import torch

def print_memory_stats():
    # System RAM
    ram = psutil.virtual_memory()
    print(f"System RAM Usage: {ram.used / 1024**3:.2f} GB / {ram.total / 1024**3:.2f} GB ({ram.percent}%)")

    # GPU RAM
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            total_gpu = torch.cuda.get_device_properties(i).total_memory
            reserved_gpu = torch.cuda.memory_reserved(i)
            allocated_gpu = torch.cuda.memory_allocated(i)
            print(f"GPU {i} Usage: {allocated_gpu / 1024**3:.2f} GB / {total_gpu / 1024**3:.2f} GB")

print_memory_stats()

System RAM Usage: 1.00 GB / 12.67 GB (10.5%)
GPU 0 Usage: 0.00 GB / 14.56 GB


In [15]:
import gc
import torch

# Clear Python and Torch memory
gc.collect()
torch.cuda.empty_cache()

# Clear Linux system cache (RAM)
!sync && echo 3 > /proc/sys/vm/drop_caches

print("System RAM and GPU memory cleared.")

/bin/bash: line 1: /proc/sys/vm/drop_caches: Read-only file system
System RAM and GPU memory cleared.


In [18]:
import torch
from safetensors.torch import load_file

def check_model_load(path="/content/ponyDiffusionV6XL.safetensors"):
    print(f"Attempting to pre-load {path} to GPU to verify memory availability...")
    try:
        # Loading with mmap=True and moving to device piece-by-piece is easier on RAM
        state_dict = load_file(path, device="cuda")
        print("Successfully loaded model to GPU VRAM.")

        # We don't actually need to keep it in memory here, we just wanted to see if it fits
        del state_dict
        torch.cuda.empty_cache()
        print("GPU Memory cleared and ready for trainer.")
    except Exception as e:
        print(f"Load failed: {e}")

check_model_load()

Attempting to pre-load /content/ponyDiffusionV6XL.safetensors to GPU to verify memory availability...
Successfully loaded model to GPU VRAM.
GPU Memory cleared and ready for trainer.


### Pro-Tip for Kohya GUI Settings:
Now that we verified the model can load, ensure these are set in the **Parameters** tab to prevent the training process itself from spiking the RAM:
1. **High-level settings**: Check `Low RAM` if available.
2. **Memory management**: Ensure `Gradient Checkpointing` is ON.
3. **Optimizers**: Use `AdamW8bit` or `Prodigy` (8-bit versions save massive amounts of RAM).

In [ ]:
!pip install --upgrade huggingface-hub accelerate

In [ ]:
!pip install --upgrade huggingface-hub accelerate

In [ ]:
!wget "https://civitai.com/api/download/models/290640" -O /content/ponyDiffusionV6XL.safetensors

In [14]:
#@title Convert Safetensors to Diffusers (Updated)
# Download a more recent version of the conversion script
!wget -q -O convert_diffusers.py https://raw.githubusercontent.com/huggingface/diffusers/main/scripts/convert_original_stable_diffusion_to_diffusers.py

# Convert Pony V6 (SDXL) to Diffusers format
# We remove the unrecognized --use_safetensors flag
!python3 convert_diffusers.py \
    --checkpoint_path /content/ponyDiffusionV6XL.safetensors \
    --dump_path /content/pony_diffusers \
    --from_safetensors \
    --to_safetensors \
    --device cuda

print("\nConversion complete! In Kohya GUI, use '/content/pony_diffusers' as your source model path.")

/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
2026-04-10 20:15:04.758845: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775852104.780318   58974 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775852104.786834   58974 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775852104.802905   58974 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:

In [ ]:
#@title Convert Safetensors to Diffusers
from_safetensors_url = '' #@param {type:"string"}
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/v0.17.1/scripts/convert_original_stable_diffusion_to_diffusers.py
!wget {from_safetensors_url} -O /content/model.safetensors
!python3 convert_original_stable_diffusion_to_diffusers.py --half --from_safetensors --checkpoint_path model.safetensors --dump_path /content/model

In [ ]:
#@title Push to HF.co

import os
from huggingface_hub import create_repo, upload_folder

hf_token = 'HF_WRITE_TOKEN' #@param {type:"string"}
repo_id = 'username/reponame' #@param {type:"string"}
commit_message = '\u2764' #@param {type:"string"}
create_repo(repo_id, private=True, token=hf_token)
model_path = '/content/train/model' #@param {type:"string"}
upload_folder(folder_path=f'{model_path}', path_in_repo='', repo_id=repo_id, commit_message=commit_message, token=hf_token)

In [ ]:
#@title Push to DagsHub.com

!pip -q install dagshub
from dagshub.upload import Repo, create_repo

repo_id = 'reponame' #@param {type:"string"}
org_name = 'orgname' #@param {type:"string"}
commit_message = '\u2764' #@param {type:"string"}
create_repo(f"{repo_id}", org_name=f"{org_name}")
repo = Repo(f"{org_name}", f"{repo_id}")
model_path = '/content/train/model' #@param {type:"string"}
repo.upload("/content/models", remote_path="data", commit_message=f"{commit_message}", force=True)